In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import metrics_search_for_fragment_df, \
metrics_search_for_two_fragments_df, widened_search_for_fragment_df, \
widened_search_for_two_fragments_df, metadata_agg_name, \
metadata_agg_subtype_name, sources_checker, \
find_aggregate_variable_names_gen_mod, \
metadata_part_name, metadata_part_subtype_name, \
find_all_variable_names_gen_mod

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [ ]:
all_data_systems = systems_cleaned[
    systems_cleaned['has_current_data']
    & systems_cleaned['has_irradiance_data']
    & systems_cleaned['has_power_data']
    & systems_cleaned['has_temperature_data']
    & systems_cleaned['has_current_data']
]
all_data_ids = set(all_data_systems.system_id)

In [4]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(
    metrics_dir,
)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [5]:
my_system_ids = list(all_data_ids.intersection(metrics_id_set))
my_system_ids.sort()
num_ids = len(my_system_ids)
num_ids

39

## DC Current Starter

In [35]:
j = 31
system_id = my_system_ids[j]
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_dc_volts = metrics_search_for_two_fragments_df(
    relevant_rows_metrics, 'dc', 'curr', 'and'
)
relevant_dc_volts

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1194,1422,4826,inv1_dc_current1,DC current,-,-,1.0,0.0,,NaN,NaN,NaN,,inv1_dc_current1__4826
1195,1422,4827,inv1_dc_current2,DC current,-,-,1.0,0.0,,NaN,NaN,NaN,,inv1_dc_current2__4827
1196,1422,4832,inv2_dc_current1,DC current,-,-,1.0,0.0,,NaN,NaN,NaN,,inv2_dc_current1__4832
1197,1422,4833,inv2_dc_current2,DC current,-,-,1.0,0.0,,NaN,NaN,NaN,,inv2_dc_current2__4833


Note: As from the first-few rich-data systems, often a `dc_pos_current` without a `dc_neg_current`.  This makes the agg-vs-parts work very interesting, as System 50 has pos and neg, and no real combination.

Also, need 'inv1' and 'inv2' as subfields, else will have serious problems!

In [32]:
dc_curr_agg_names = ['dc_current', 'input_current', 'dc_current_l2']

In [75]:
def find_aggregate_dc_current(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    sources_matter: bool = True,
    known_sources=('inverter', 'module'),
    known_sources_short=('inv', 'mod')
):
    var_name = 'dc_current'
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'dc', 'curr', 'and'
    )
    dc_curr_agg_names = ['dc_current', 'input_current', 'dc_current_I2', 'Battery_A_Avg']
    return find_aggregate_variable_names_gen_mod(
        systems_cleaned=systems_cleaned,
        filtered_metrics_df=filtered_metrics_df,
        var_name='dc_current',
        agg_var_sensor_names=dc_curr_agg_names,
        print_messages=print_messages,
        sources_matter=sources_matter,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )

In [76]:
all_agg_dc_curr, all_agg_dc_curr_meta = find_aggregate_dc_current(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=True,
    sources_matter=True
)

System 2 appears to have no obvious dc_current aggregator name.
System 3 appears to have no obvious dc_current aggregator name.
System 1283 appears to have no obvious dc_current aggregator name.
System 1284 appears to have no obvious dc_current aggregator name.
System 4 appears to have no obvious dc_current aggregator name.
System 1416 appears to have no obvious dc_current aggregator name.
System 1289 appears to have no obvious dc_current aggregator name.
System 10 appears to have no obvious dc_current aggregator name.
System 1419 appears to have no obvious dc_current aggregator name.
System 1422 appears to have no obvious dc_current aggregator name.
System 1423 appears to have no obvious dc_current aggregator name.
System 1429 appears to have no obvious dc_current aggregator name.
System 1305 appears to have no obvious dc_current aggregator name.
System 1306 appears to have no obvious dc_current aggregator name.
System 1307 appears to have no obvious dc_current aggregator name.
System

In [ ]:
def find_all_dc_current(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    sources_matter: bool = True,
    known_sources=('inverter', 'module'),
    known_sources_short=('inv', 'mod')
):
    var_name = 'dc_current'
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'dc', 'curr', 'and'
    )
    dc_curr_agg_names = ['dc_current', 'input_current', 'dc_current_I2', 'Battery_A_Avg']
    all_agg_dc_cur, all_agg_dc_cur_meta = find_aggregate_dc_current(
        systems_cleaned=systems_cleaned,
        metrics_df=metrics_df,
        print_messages=print_messages,
        sources_matter=sources_matter,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    # Requires the modified function from the updated
    # gen_variable_standard_static.py
    all_ac_cur, all_ac_cur_metadata = find_all_variable_names_gen_mod(
            var_aggs_dict=all_agg_dc_cur,
            var_aggs_metadata=all_agg_dc_cur_meta,
            filtered_metrics_df=filtered_metrics_df,
            var_name=var_name,
            agg_var_sensor_names=dc_curr_agg_names,
            sources_matter=sources_matter,
            known_sources=known_sources,
            known_sources_short=known_sources_short,
            hard_stop_on_singleton=False
        )
    # cleanup -- if solo 'positive' part-current (no negative),
    # go ahead and upgrade it to an aggregate
    for system_id in all_ac_cur.keys():
        num_parts = 0
        num_pos_parts = 0
        num_neg_parts = 0
        for metric_dict in all_ac_cur[system_id]:
            if metric_dict['whole_or_part'] == 'part':
                num_parts += 1
                if 'pos' in metric_dict['sensor_name']:
                    num_pos_parts += 1
                elif 'neg' in metric_dict['sensor_name']:
                    num_neg_parts += 1
        if (num_neg_parts == 0) and (num_pos_parts == 1):
            # unique positive part upgraded to an aggregate
            all_ac_cur_metadata.loc[
                system_id, metadata_agg_name(var_name=var_name)
            ] = True
            if num_parts == 1:
                all_ac_cur_metadata.loc[
                    system_id, metadata_part_name(var_name=var_name)
                ] = False
            for metric_dict in all_ac_cur[system_id]:
                if (
                    (metric_dict['whole_or_part'] == 'part')
                    and ('pos' in metric_dict['sensor_name'])
                ):
                    metric_dict['whole_or_part'] = 'whole'
                    metric_dict.pop('index')
                    if 'source' in metric_dict:
                        all_ac_cur_metadata.loc[
                            system_id, metadata_agg_subtype_name(
                                var_name, metric_dict['source_type']
                            )
                        ] = True
                        if num_parts == 1:
                            all_ac_cur_metadata.loc[
                                system_id, 
                                metadata_part_subtype_name(
                                    var_name, metric_dict['source_type']
                                )
                            ] = False
                    # only one positive part by hypothesis, so
                    break
        elif num_neg_parts < num_pos_parts:
            print(f'For System {system_id}, an awkward mismatch between parts.\n'
                  + f'Positive parts: {num_pos_parts}, Negative parts: {num_neg_parts}.\n'
                  + 'Check and see if all of the output looks right.')
    return (all_ac_cur, all_ac_cur_metadata)
    

In [78]:
all_dc_curr, all_dc_curr_metadata = find_all_dc_current(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=False,
    sources_matter=True
)

System 2 has only one subpart for dc_current!
system_id                             2
metric_id                           348
sensor_name              dc_pos_current
common_name                  DC current
raw_units                             A
units                                 A
calc_scale                          1.0
calc_offset                         0.0
calc_details                           
aggregation_type                    avg
source_type                         NaN
source_id                           NaN
comments                               
standard_name       dc_pos_current__348
Name: 1298, dtype: object
0
0
1
System 3 has only one subpart for dc_current!
system_id                             3
metric_id                           355
sensor_name              dc_pos_current
common_name                  DC current
raw_units                             A
units                                 A
calc_scale                          1.0
calc_offset                         

In [92]:
all_dc_curr[1429]

[{'metric_id': np.int32(4913),
  'sensor_name': 'inv1_dc_current1',
  'common_name': 'DC current',
  'units': '-',
  'calc_details': '',
  'whole_or_part': 'part',
  'source_type': 'inverter',
  'index': '1_dc_current1'},
 {'metric_id': np.int32(4914),
  'sensor_name': 'inv1_dc_current2',
  'common_name': 'DC current',
  'units': '-',
  'calc_details': '',
  'whole_or_part': 'part',
  'source_type': 'inverter',
  'index': '1_dc_current2'},
 {'metric_id': np.int32(4919),
  'sensor_name': 'inv2_dc_current1',
  'common_name': 'DC current',
  'units': '-',
  'calc_details': '',
  'whole_or_part': 'part',
  'source_type': 'inverter',
  'index': '2_dc_current1'},
 {'metric_id': np.int32(4920),
  'sensor_name': 'inv2_dc_current2',
  'common_name': 'DC current',
  'units': '-',
  'calc_details': '',
  'whole_or_part': 'part',
  'source_type': 'inverter',
  'index': '2_dc_current2'}]

Several two-category divisions!  Very annoying!

In [80]:
part_metric_names = []
for system_id in all_dc_curr.keys():
    for metric_dict in all_dc_curr[system_id]:
        if metric_dict['whole_or_part'] == 'part':
            part_metric_names.append(metric_dict['sensor_name'])
part_metric_names.sort()
for metric_name in part_metric_names:
    print(metric_name)

ShuntCurrent_A_Avg_1
ShuntCurrent_A_Avg_1
ShuntCurrent_A_Avg_1
ShuntCurrent_A_Avg_2
ShuntCurrent_A_Avg_2
ShuntCurrent_A_Avg_2
ShuntCurrent_A_Avg_3
ShuntCurrent_A_Avg_3
ShuntCurrent_A_Avg_3
ShuntCurrent_A_Avg_4
ShuntCurrent_A_Avg_4
ShuntCurrent_A_Avg_4
ShuntCurrent_A_Avg_5
ShuntCurrent_A_Avg_5
ShuntCurrent_A_Avg_6
ShuntCurrent_A_Avg_6
ShuntCurrent_A_Avg_7
ShuntCurrent_A_Avg_7
String11Current_shaded
String12Current_shaded
String1Current_unshaded
String2Current_unshaded
dc_current_1
dc_current_1
dc_current_1
dc_current_1
dc_current_1
dc_current_1
dc_current_1_Idc
dc_current_2
dc_current_2
dc_current_2
dc_current_2
dc_current_2
dc_current_2
dc_current_2_Idc
dc_current_3
dc_current_3
dc_current_3
dc_current_3
dc_current_3
dc_current_3
dc_current_4
dc_current_4
dc_current_5
dc_current_5
dc_current_6
dc_current_6
dc_current_negative
dc_current_positive
dc_neg_current
dc_neg_current
dc_pos_current
dc_pos_current
inv1_dc_current
inv1_dc_current
inv1_dc_current
inv1_dc_current
inv1_dc_current
in

In [83]:
units_col = []
for system_id in all_dc_curr.keys():
    for metric_dict in all_dc_curr[system_id]:
        units_col.append(metric_dict['units'])
units_col = set(units_col)
for unit in units_col:
    print(unit)

A
-


In [88]:
weird_systems = []
for system_id in all_dc_curr.keys():
    for metric_dict in all_dc_curr[system_id]:
        if metric_dict['units'] == '-':
            weird_systems.append(system_id)
weird_systems = set(weird_systems)
for system_id in weird_systems:
    print(system_id)

1403
1429
1422
1423


All systems affected have missing subparts and no units whatsoever.  Will do a quick check, but as long as the scale is reasonable, assumed amperes.

Ok, so we dropped the errors, and 'promoted' dc_pos_current to aggregate successfully.  But still need to resolve 4901-4903 issues of which term is important to avoid name-repeats.

## AC Current Starter

In [132]:
j = 2
system_id = my_system_ids[j]
# system_id = 1214
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_ac_volts = metrics_search_for_two_fragments_df(
    relevant_rows_metrics, 'ac', 'curr', 'and'
)
relevant_ac_volts

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1713,4,319,ac_current,AC current,A,A,1.0,0.0,,avg,NaN,NaN,,ac_current__319


In [133]:
relevant_ac_volts['sensor_name']

1713    ac_current
Name: sensor_name, dtype: str

Immediate observation: battery or no battery, sources matter!
Still lots of missingness for aggregates!

In [ ]:
ac_current_agg_names = [
    'ac_current',
    'ac_current_Iac',
    'InvIabcAvg_Avg',
    'InvINeutral_Avg',
]

In [134]:
def find_aggregate_ac_current(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    known_sources=('inverter', 'meter'),
    known_sources_short=('inv', 'met')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'ac', 'curr', 'and'
    )
    ac_current_agg_names = [
        'ac_current',
        'ac_current_Iac',
        'InvIabcAvg_Avg',
        'InvINeutral_Avg',
    ]
    return find_aggregate_variable_names_gen_mod(
        systems_cleaned=systems_cleaned,
        filtered_metrics_df=filtered_metrics_df,
        var_name='ac_current',
        agg_var_sensor_names=ac_current_agg_names,
        print_messages=print_messages,
        sources_matter=True,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )


In [135]:
all_agg_ac_current, all_agg_ac_current_metadata = find_aggregate_ac_current(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=True
)

System 1203 appears to have no obvious ac_current aggregator name.
System 1214 appears to have no obvious ac_current aggregator name.
System 1216 appears to have no obvious ac_current aggregator name.
System 1217 appears to have no obvious ac_current aggregator name.
System 1218 appears to have no obvious ac_current aggregator name.
System 1219 appears to have no obvious ac_current aggregator name.
System 1220 appears to have no obvious ac_current aggregator name.
System 1221 appears to have no obvious ac_current aggregator name.
System 1222 appears to have no obvious ac_current aggregator name.
System 1223 appears to have no obvious ac_current aggregator name.
System 1224 appears to have no obvious ac_current aggregator name.
System 1225 appears to have no obvious ac_current aggregator name.
System 1226 appears to have no obvious ac_current aggregator name.
System 1244 appears to have no obvious ac_current aggregator name.
System 1245 appears to have no obvious ac_current aggregator n

In [138]:
def find_all_ac_current(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    known_sources=('inverter', 'meter'),
    known_sources_short=('inv', 'met')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'ac', 'curr', 'and'
    )
    # no percentages
    filtered_metrics_df = filtered_metrics_df[filtered_metrics_df['units'] == 'A']
    ac_current_agg_names = [
        'ac_current',
        'ac_current_Iac',
        'InvIabcAvg_Avg',
        'InvINeutral_Avg',
        'A_avg'
    ]
    all_agg_ac_cur, all_agg_ac_cur_metadata = find_aggregate_ac_current(
        systems_cleaned=systems_cleaned,
        metrics_df=metrics_df,
        print_messages=print_messages,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    return find_all_variable_names_gen_mod(
        var_aggs_dict=all_agg_ac_cur,
        var_aggs_metadata=all_agg_ac_cur_metadata,
        filtered_metrics_df=filtered_metrics_df,
        var_name='ac_current',
        agg_var_sensor_names=ac_current_agg_names,
        sources_matter=True,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )


In [139]:
all_ac_curr, all_ac__curr_metadata = find_all_ac_current(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=False,
)

In [140]:
units_list = []
for system_id in all_ac_curr.keys():
    for metric_dict in all_ac_curr[system_id]:
        units_list.append(metric_dict['units'])
units_list = set(units_list)
units_list
    

{'A'}

In [141]:
part_metric_names = []
for system_id in all_ac_curr.keys():
    for metric_dict in all_ac_curr[system_id]:
        if metric_dict['whole_or_part'] == 'part':
            part_metric_names.append(metric_dict['sensor_name'])
part_metric_names.sort()
for metric_name in part_metric_names:
    print(metric_name)

InvIa_Avg
InvIa_Avg
InvIa_Avg
InvIb_Avg
InvIb_Avg
InvIb_Avg
InvIc_Avg
InvIc_Avg
InvIc_Avg
ac_current_1
ac_current_1
ac_current_1
ac_current_1
ac_current_2
ac_current_2
ac_current_2
ac_current_2
ac_current_3
ac_current_3
ac_current_3
ac_current_3
ac_current_phA
ac_current_phB
ac_current_phC
inv1_ac_current
inv1_ac_current
inv1_ac_current
inv1_ac_current
inv1_ac_current
inv1_ac_current
inv1_ac_current
inv1_ac_current
inv2_ac_current
inv2_ac_current
inv2_ac_current
inv2_ac_current
inv2_ac_current
inv2_ac_current
inv2_ac_current
inv2_ac_current
inv3_ac_current
inv3_ac_current
inv4_ac_current
inv4_ac_current
inv5_ac_current
inv5_ac_current
inv6_ac_current
inv6_ac_current
inv7_ac_current
inv8_ac_current
inv9_ac_current


AC CUrrent is regular!